# 02 · Pilot Collection: 2024 Season

**Blueprint step:** Step 4 (pilot, as recommended in "What we should do first", action 5) · **Checkpoint:** 2 (pilot part)

**Purpose:** run the complete collection pipeline end-to-end for **one season (2024)** and reconcile every record, before scaling to 1950–2025.

**How it works:** all data comes through the cached client `src/api_client.py` (Step 3); this notebook never calls `requests` directly. Season-level endpoints are paged with `limit=100` (blueprint §15). `src/collect.py` flattens the JSON into tables and keeps **the API's raw text values**: no type conversion, no cleaning, no features, no target variables.

**Not done here:** 1950–2025 collection, weather, cleaning, final DNF/DSQ/non-starter rules, feature engineering, splits, models.

**Re-running:** after the first run every response comes from the cache in `data/raw/jolpica/`, so a re-run makes **0** API requests.

## 1. Configuration

In [1]:
import hashlib
import io
import logging
import sys
from pathlib import Path

import pandas as pd

# Project root = folder containing config.yaml (works from any start folder)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("config.yaml not found - open this notebook inside the project")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))  # so "import src..." works from notebooks/

from importlib.metadata import version
pins = dict(line.split("==") for line in (PROJECT_ROOT / "requirements.txt").read_text().splitlines()
            if "==" in line and not line.startswith("#"))
for package in ["pandas", "requests", "PyYAML"]:
    assert version(package) == pins[package], "select the 'Python 3.11 (f1-race-prediction)' kernel"

from src.api_client import JolpicaClient
from src.collect import collect_season, flatten_results, flatten_qualifying, race_summary
from src.config import load_config

# Show only warnings/errors from the client (retries, HTTP errors); counts are shown via client.stats
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

CONFIG = load_config()
SEASON = 2024  # pilot season (the full 1950-2025 range in config.yaml is used later)
INTERIM_DIR = PROJECT_ROOT / CONFIG["paths"]["data_interim"]
TABLES_DIR = PROJECT_ROOT / CONFIG["paths"]["tables"]
RAW_JOLPICA = PROJECT_ROOT / CONFIG["paths"]["data_raw_jolpica"]
MANIFEST = PROJECT_ROOT / CONFIG["paths"]["raw_manifest"]

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)

def snapshot_raw():
    # sha256 of every cached raw file, to prove later that nothing was overwritten
    return {p.relative_to(PROJECT_ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in RAW_JOLPICA.rglob("*.json")} if RAW_JOLPICA.exists() else {}

raw_before = snapshot_raw()
manifest_rows_before = len(pd.read_csv(MANIFEST)) if MANIFEST.exists() else 0

client = JolpicaClient()
print("Season:", SEASON, "| base URL:", client.base_url, "| page size:", client.page_limit)
print("Cached raw files before this run:", len(raw_before), "| manifest rows:", manifest_rows_before)

Season: 2024 | base URL: https://api.jolpi.ca/ergast/f1 | page size: 100
Cached raw files before this run: 18 | manifest rows: 18


## 2. Discover the 2024 race calendar

The race list comes from the API (`/2024/races.json`); the number of races is **not** hard-coded. `collect_season()` fetches the calendar, results, qualifying, sprint and the season's driver and constructor lists, all through the cached client.

In [2]:
tables = collect_season(client, SEASON)
races, results, qualifying, sprints = tables["races"], tables["results"], tables["qualifying"], tables["sprints"]
drivers, constructors = tables["drivers"], tables["constructors"]

print("Client activity:", client.stats)
for name, df in tables.items():
    print(f"  {name:<13} {df.shape[0]:>4} rows x {df.shape[1]} columns")

Client activity: {'cache_hits': 15, 'network_requests': 0, 'retries': 0}
  races           24 rows x 13 columns
  results        479 rows x 34 columns
  qualifying     479 rows x 25 columns
  sprints        120 rows x 34 columns
  drivers         25 rows x 8 columns
  constructors    10 rows x 4 columns


In [3]:
calendar_total = int(client.get_json(f"/{SEASON}/races.json", {"limit": 100, "offset": 0})["MRData"]["total"])  # cache hit
rounds = races["round"].astype(int)

print("Races in API calendar (MRData.total):", calendar_total)
print("Races in our table                  :", len(races))
print("Rounds are 1..N in order            :", rounds.tolist() == list(range(1, len(races) + 1)))
print("Duplicate rounds                    :", rounds.duplicated().sum())
print("Duplicate race_id                   :", races["race_id"].duplicated().sum())
print("Missing race_time                   :", races["race_time"].isna().sum())
print("Sprint weekends (calendar)          :", races.loc[races["sprint_scheduled"], "round"].astype(int).tolist())
races

Races in API calendar (MRData.total): 24
Races in our table                  : 24
Rounds are 1..N in order            : True
Duplicate rounds                    : 0
Duplicate race_id                   : 0
Missing race_time                   : 0
Sprint weekends (calendar)          : [5, 6, 11, 19, 21, 23]


,season,round,race_id,race_name,race_date,race_time,circuit_id,circuit_name,circuit_lat,circuit_lon,locality,country,sprint_scheduled
0,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,False
1,2024,2,2024_02,Saudi Arabian Grand Prix,2024-03-09,17:00:00Z,jeddah,Jeddah Corniche Circuit,21.6319,39.1044,Jeddah,Saudi Arabia,False
2,2024,3,2024_03,Australian Grand Prix,2024-03-24,04:00:00Z,albert_park,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia,False
3,2024,4,2024_04,Japanese Grand Prix,2024-04-07,05:00:00Z,suzuka,Suzuka Circuit,34.8431,136.541,Suzuka,Japan,False
4,2024,5,2024_05,Chinese Grand Prix,2024-04-21,07:00:00Z,shanghai,Shanghai International Circuit,31.3389,121.22,Shanghai,China,True
5,2024,6,2024_06,Miami Grand Prix,2024-05-05,20:00:00Z,miami,Miami International Autodrome,25.9581,-80.2389,Miami,USA,True
6,2024,7,2024_07,Emilia Romagna Grand Prix,2024-05-19,13:00:00Z,imola,Autodromo Enzo e Dino Ferrari,44.3439,11.7167,Imola,Italy,False
7,2024,8,2024_08,Monaco Grand Prix,2024-05-26,13:00:00Z,monaco,Circuit de Monaco,43.7347,7.42056,Monte Carlo,Monaco,False
8,2024,9,2024_09,Canadian Grand Prix,2024-06-09,18:00:00Z,villeneuve,Circuit Gilles Villeneuve,45.5,-73.5228,Montreal,Canada,False
9,2024,10,2024_10,Spanish Grand Prix,2024-06-23,13:00:00Z,catalunya,Circuit de Barcelona-Catalunya,41.57,2.26111,Barcelona,Spain,False


### 2.1 Season driver and constructor lists

In [4]:
print(f"Drivers listed for {SEASON}:", len(drivers), "| constructors:", len(constructors))
print("Duplicate driver_id:", drivers["driver_id"].duplicated().sum(),
      "| duplicate constructor_id:", constructors["constructor_id"].duplicated().sum())
constructors

Drivers listed for 2024: 25 | constructors: 10
Duplicate driver_id: 0 | duplicate constructor_id: 0


,season,constructor_id,constructor_name,constructor_nationality
0,2024,alpine,Alpine F1 Team,French
1,2024,aston_martin,Aston Martin,British
2,2024,ferrari,Ferrari,Italian
3,2024,haas,Haas F1 Team,American
4,2024,mclaren,McLaren,British
5,2024,mercedes,Mercedes,German
6,2024,rb,RB F1 Team,Italian
7,2024,red_bull,Red Bull,Austrian
8,2024,sauber,Sauber,Swiss
9,2024,williams,Williams,British


## 3. Race results

In [5]:
results_total = int(client.get_json(f"/{SEASON}/results.json", {"limit": 100, "offset": 0})["MRData"]["total"])
result_pages = sorted(RAW_JOLPICA.glob(f"{SEASON}/results__limit-100__offset-*.json"))
print("MRData.total (season results):", results_total)
print("Rows collected               :", len(results), "| equal:", len(results) == results_total)
print("Cached result pages          :", [p.name for p in result_pages])
print("Columns:", list(results.columns))
print("\nAll values are kept as text (types):", sorted({type(v).__name__ for v in results['position']}))
results.head(3)

MRData.total (season results): 479
Rows collected               : 479 | equal: True
Cached result pages          : ['results__limit-100__offset-0.json', 'results__limit-100__offset-100.json', 'results__limit-100__offset-200.json', 'results__limit-100__offset-300.json', 'results__limit-100__offset-400.json']
Columns: ['season', 'round', 'race_id', 'race_name', 'race_date', 'race_time', 'circuit_id', 'circuit_name', 'locality', 'country', 'driver_id', 'driver_code', 'permanent_number', 'driver_number', 'driver_first_name', 'driver_last_name', 'driver_dob', 'driver_nationality', 'constructor_id', 'constructor_name', 'constructor_nationality', 'grid', 'position', 'position_text', 'points', 'laps', 'status', 'time_millis', 'time_text', 'fastest_lap_rank', 'fastest_lap_lap', 'fastest_lap_time', 'fastest_lap_avg_speed', 'fastest_lap_speed_units']

All values are kept as text (types): ['str']


,season,round,race_id,race_name,race_date,race_time,circuit_id,circuit_name,locality,country,driver_id,driver_code,permanent_number,driver_number,driver_first_name,driver_last_name,driver_dob,driver_nationality,constructor_id,constructor_name,constructor_nationality,grid,position,position_text,points,laps,status,time_millis,time_text,fastest_lap_rank,fastest_lap_lap,fastest_lap_time,fastest_lap_avg_speed,fastest_lap_speed_units
0,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,max_verstappen,VER,3,1,Max,Verstappen,1997-09-30,Dutch,red_bull,Red Bull,Austrian,1,1,1,26,57,Finished,5504742,1:31:44.742,1,39,1:32.608,210.383,kph
1,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,perez,PER,11,11,Sergio,Pérez,1990-01-26,Mexican,red_bull,Red Bull,Austrian,5,2,2,18,57,Finished,5527199,+22.457,4,40,1:34.364,206.468,kph
2,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,sainz,SAI,55,55,Carlos,Sainz,1994-09-01,Spanish,ferrari,Ferrari,Italian,4,3,3,15,57,Finished,5529852,+25.110,6,44,1:34.507,206.156,kph


## 4. Qualifying

In [6]:
quali_total = int(client.get_json(f"/{SEASON}/qualifying.json", {"limit": 100, "offset": 0})["MRData"]["total"])
print("MRData.total (season qualifying):", quali_total, "| rows collected:", len(qualifying),
      "| equal:", len(qualifying) == quali_total)
print("Races with qualifying data:", qualifying["round"].nunique(), "of", len(races))
print("Missing Q1 / Q2 / Q3 (field absent in API):",
      qualifying["q1"].isna().sum(), "/", qualifying["q2"].isna().sum(), "/", qualifying["q3"].isna().sum())
qualifying.head(3)

MRData.total (season qualifying): 479 | rows collected: 479 | equal: True
Races with qualifying data: 24 of 24
Missing Q1 / Q2 / Q3 (field absent in API): 0 / 119 / 239


,season,round,race_id,race_name,race_date,race_time,circuit_id,circuit_name,locality,country,driver_id,driver_code,permanent_number,driver_number,driver_first_name,driver_last_name,driver_dob,driver_nationality,constructor_id,constructor_name,constructor_nationality,qualifying_position,q1,q2,q3
0,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,max_verstappen,VER,3,1,Max,Verstappen,1997-09-30,Dutch,red_bull,Red Bull,Austrian,1,1:30.031,1:29.374,1:29.179
1,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,leclerc,LEC,16,16,Charles,Leclerc,1997-10-16,Monegasque,ferrari,Ferrari,Italian,2,1:30.243,1:29.165,1:29.407
2,2024,1,2024_01,Bahrain Grand Prix,2024-03-02,15:00:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,russell,RUS,63,63,George,Russell,1998-02-15,British,mercedes,Mercedes,German,3,1:30.350,1:29.922,1:29.485


## 5. Sprint results

Sprint rounds are **found from the data**, not assumed: we compare the calendar's `Sprint` sessions with the rounds that actually have sprint results.

In [7]:
sprint_total = int(client.get_json(f"/{SEASON}/sprint.json", {"limit": 100, "offset": 0})["MRData"]["total"])
scheduled_sprints = sorted(races.loc[races["sprint_scheduled"], "round"].astype(int))
rounds_with_sprint_results = sorted(int(r) for r in sprints["round"].unique())
print("MRData.total (season sprint):", sprint_total, "| rows collected:", len(sprints),
      "| equal:", len(sprints) == sprint_total)
print("Sprint sessions in calendar :", scheduled_sprints)
print("Rounds with sprint results  :", rounds_with_sprint_results)
print("Calendar and results agree  :", scheduled_sprints == rounds_with_sprint_results)
sprints[["round", "race_name", "driver_id", "constructor_id", "grid", "position", "position_text",
         "points", "laps", "status", "time_text", "fastest_lap_rank"]].head(3)

MRData.total (season sprint): 120 | rows collected: 120 | equal: True
Sprint sessions in calendar : [5, 6, 11, 19, 21, 23]
Rounds with sprint results  : [5, 6, 11, 19, 21, 23]
Calendar and results agree  : True


,round,race_name,driver_id,constructor_id,grid,position,position_text,points,laps,status,time_text,fastest_lap_rank
0,5,Chinese Grand Prix,max_verstappen,red_bull,4,1,1,8,19,Finished,32:04.660,1
1,5,Chinese Grand Prix,hamilton,mercedes,2,2,2,7,19,Finished,+13.043,2
2,5,Chinese Grand Prix,perez,red_bull,6,3,3,6,19,Finished,+15.258,7


## 6. Reconcile records

### 6.1 Race-level record counts

In [8]:
summary = race_summary(tables)
summary

,season,round,race_id,race_name,number_of_results,number_of_qualifying_records,number_of_sprint_records,duplicate_result_keys,missing_driver_ids,missing_constructor_ids
0,2024,1,2024_01,Bahrain Grand Prix,20,20,0,0,0,0
1,2024,2,2024_02,Saudi Arabian Grand Prix,20,20,0,0,0,0
2,2024,3,2024_03,Australian Grand Prix,19,19,0,0,0,0
3,2024,4,2024_04,Japanese Grand Prix,20,20,0,0,0,0
4,2024,5,2024_05,Chinese Grand Prix,20,20,20,0,0,0
5,2024,6,2024_06,Miami Grand Prix,20,20,20,0,0,0
6,2024,7,2024_07,Emilia Romagna Grand Prix,20,20,0,0,0,0
7,2024,8,2024_08,Monaco Grand Prix,20,20,0,0,0,0
8,2024,9,2024_09,Canadian Grand Prix,20,20,0,0,0,0
9,2024,10,2024_10,Spanish Grand Prix,20,20,0,0,0,0


In [9]:
totals = summary[[c for c in summary.columns if c.startswith(("number_", "duplicate_", "missing_"))]].sum()
print("Overall totals:")
print(totals.to_string())

Overall totals:
number_of_results               479
number_of_qualifying_records    479
number_of_sprint_records        120
duplicate_result_keys             0
missing_driver_ids                0
missing_constructor_ids           0


### 6.2 Keys and cross-source checks

In [10]:
KEY = ["season", "round", "driver_id"]
for name, df in [("results", results), ("qualifying", qualifying), ("sprints", sprints)]:
    dup = df[df.duplicated(KEY, keep=False)]
    print(f"{name:<10} rows={len(df):<4} duplicate (season, round, driver_id) rows={len(dup)}")
    if len(dup):
        display(dup[KEY + ["position", "position_text", "status", "constructor_id"]])

print("\nA. Race count          : calendar", len(races), "| results", results["round"].nunique(),
      "| qualifying", qualifying["round"].nunique(), "| sprint", sprints["round"].nunique())
print("E. Unique race keys     :", races[["season", "round"]].drop_duplicates().shape[0] == len(races))
extra = set(results["round"]) - set(races["round"])
print("Result rounds not in calendar:", sorted(extra, key=int) or "none")
print("Calendar rounds without results:", sorted(set(races["round"]) - set(results["round"]), key=int) or "none")

results    rows=479  duplicate (season, round, driver_id) rows=0
qualifying rows=479  duplicate (season, round, driver_id) rows=0
sprints    rows=120  duplicate (season, round, driver_id) rows=0

A. Race count          : calendar 24 | results 24 | qualifying 24 | sprint 6
E. Unique race keys     : True
Result rounds not in calendar: none
Calendar rounds without results: none


In [11]:
# Drivers / constructors: race results vs the season lists
print("Drivers in results but not in season list :", sorted(set(results["driver_id"]) - set(drivers["driver_id"])) or "none")
print("Drivers in season list but not in results :", sorted(set(drivers["driver_id"]) - set(results["driver_id"])) or "none")
print("Constructors in results but not in list   :", sorted(set(results["constructor_id"]) - set(constructors["constructor_id"])) or "none")
print("Constructors in list but not in results   :", sorted(set(constructors["constructor_id"]) - set(results["constructor_id"])) or "none")

# Per race: drivers in results vs qualifying
rows = []
for rnd, g in results.groupby("round"):
    q = set(qualifying.loc[qualifying["round"] == rnd, "driver_id"])
    r = set(g["driver_id"])
    if q != r:
        rows.append({"round": int(rnd), "in_results_not_qualifying": sorted(r - q),
                     "in_qualifying_not_results": sorted(q - r)})
print("\nRaces where the qualifying driver set differs from the results driver set:", len(rows))
pd.DataFrame(rows)

Drivers in results but not in season list : none
Drivers in season list but not in results : ['antonelli']
Constructors in results but not in list   : none
Constructors in list but not in results   : none

Races where the qualifying driver set differs from the results driver set: 0


""


### 6.3 Cross-check with the per-round cache from Step 3

Step 3 cached `/2024/1/results`, `/2024/1/qualifying` and `/2024/5/sprint` as **per-round** requests. The season-level (paginated) data must give exactly the same rows for those races. These calls are cache hits: 0 new requests.

In [12]:
per_round = {
    "results R1": (flatten_results(client.get_race_results(SEASON, 1)), results[results["round"] == "1"]),
    "qualifying R1": (flatten_qualifying(client.get_qualifying(SEASON, 1)), qualifying[qualifying["round"] == "1"]),
    "sprint R5": (flatten_results(client.get_sprint_results(SEASON, 5), "SprintResults"), sprints[sprints["round"] == "5"]),
}
for name, (single, from_season) in per_round.items():
    same = single.reset_index(drop=True).equals(from_season.reset_index(drop=True))
    print(f"{name:<14} per-round rows={len(single):<3} season rows={len(from_season):<3} identical={same}")

results R1     per-round rows=20  season rows=20  identical=True
qualifying R1  per-round rows=20  season rows=20  identical=True
sprint R5      per-round rows=20  season rows=20  identical=True


## 7. Basic data-quality checks

Numeric checks use **temporary** numeric copies (`pd.to_numeric`) inside this section only; the stored tables stay as text. **No DNF rule is decided here.** We only report what the data shows, so the rule can be finalised at Step 7 (validation).

In [13]:
missing = pd.DataFrame({name: df[["race_id", "driver_id", "constructor_id"]].isna().sum()
                        for name, df in [("results", results), ("qualifying", qualifying), ("sprints", sprints)]})
print("Missing IDs per table:")
print(missing.to_string())

counts = summary["number_of_results"]
print("\nDrivers per race (results):", counts.value_counts().sort_index().to_dict())
print("Races with an unusual driver count:", summary.loc[counts != counts.mode()[0], ["round", "race_name", "number_of_results"]].to_dict("records") or "none")

Missing IDs per table:
                results  qualifying  sprints
race_id               0           0        0
driver_id             0           0        0
constructor_id        0           0        0

Drivers per race (results): {19: 1, 20: 23}
Races with an unusual driver count: [{'round': '3', 'race_name': 'Australian Grand Prix', 'number_of_results': 19}]


In [14]:
checks = []
for name, df in [("results", results), ("sprints", sprints)]:
    pos = pd.to_numeric(df["position"], errors="coerce")
    grid = pd.to_numeric(df["grid"], errors="coerce")
    n_in_race = df.groupby("round")["driver_id"].transform("size")
    checks.append({
        "table": name,
        "position not numeric": int(pos.isna().sum()),
        "position outside 1..N": int(((pos < 1) | (pos > n_in_race)).sum()),
        "races with repeated positions": int(df.assign(p=pos).duplicated(["round", "p"]).groupby(df["round"]).any().sum()),
        "grid not numeric": int(grid.isna().sum()),
        "grid outside 0..N": int(((grid < 0) | (grid > n_in_race)).sum()),
        "grid = 0": int((grid == 0).sum()),
    })
pd.DataFrame(checks)

,table,position not numeric,position outside 1..N,races with repeated positions,grid not numeric,grid outside 0..N,grid = 0
0,results,0,0,0,0,0,1
1,sprints,0,0,0,0,0,0


In [15]:
print("grid = 0 rows (race results):")
results[pd.to_numeric(results["grid"]) == 0][["round", "race_name", "driver_id", "constructor_id", "grid",
                                              "position", "position_text", "laps", "status"]]

grid = 0 rows (race results):


,round,race_name,driver_id,constructor_id,grid,position,position_text,laps,status
391,20,Mexico City Grand Prix,ocon,alpine,0,13,13,70,Lapped


### 7.1 Observed status / positionText combinations (input for the Step 7 DNF decision)

In [16]:
both = pd.concat([results.assign(source="race"), sprints.assign(source="sprint")], ignore_index=True)
both["position_text_type"] = both["position_text"].where(~both["position_text"].str.isdigit(), "numeric")
status_combinations = (both.groupby(["source", "status", "position_text_type"]).size()
                       .rename("records").reset_index().sort_values(["source", "records"], ascending=[True, False]))
status_combinations

,source,status,position_text_type,records
3,race,Finished,numeric,287
4,race,Lapped,numeric,138
5,race,Retired,R,43
6,race,Retired,numeric,6
0,race,Did not start,W,3
1,race,Disqualified,D,1
2,race,Disqualified,numeric,1
7,sprint,Finished,numeric,116
8,sprint,Retired,R,3
9,sprint,Retired,numeric,1


In [17]:
special = both[both["status"].isin(["Disqualified", "Did not start"]) |
               ((both["status"] == "Retired") & (both["position_text_type"] == "numeric")) |
               (both["position_text_type"] != "numeric")]
print("Non-numeric positionText by source:",
      both[both["position_text_type"] != "numeric"].groupby(["source", "position_text"]).size().to_dict())
print("Disqualified:", (both["status"] == "Disqualified").sum(),
      "| Did not start:", (both["status"] == "Did not start").sum(),
      "| Retired but classified (numeric positionText):",
      ((both["status"] == "Retired") & (both["position_text_type"] == "numeric")).sum())
special[special["status"] != "Retired"][["source", "round", "race_name", "driver_id", "grid", "position",
                                         "position_text", "laps", "status"]]

Non-numeric positionText by source: {('race', 'D'): 1, ('race', 'R'): 43, ('race', 'W'): 3, ('sprint', 'R'): 3}
Disqualified: 2 | Did not start: 3 | Retired but classified (numeric positionText): 7


,source,round,race_name,driver_id,grid,position,position_text,laps,status
238,race,12,British Grand Prix,gasly,19,20,W,0,Did not start
278,race,14,Belgian Grand Prix,russell,6,20,20,44,Disqualified
416,race,21,São Paulo Grand Prix,albon,7,18,W,0,Did not start
417,race,21,São Paulo Grand Prix,stroll,10,19,W,0,Did not start
418,race,21,São Paulo Grand Prix,hulkenberg,18,20,D,0,Disqualified


### 7.2 Qualifying times

In [18]:
q_missing = qualifying.groupby("round")[["q1", "q2", "q3"]].apply(lambda d: d.isna().sum())
q_missing.index = q_missing.index.astype(int)
q_missing = q_missing.sort_index()
print("Rows with missing Q1 (no Q1 time):", qualifying["q1"].isna().sum())
display(qualifying[qualifying["q1"].isna()][["round", "race_name", "driver_id", "qualifying_position", "q1", "q2", "q3"]])
print("Missing Q2 / Q3 per race (expected about 5 / 10 in the knock-out format):")
q_missing.T

Rows with missing Q1 (no Q1 time): 0


,round,race_name,driver_id,qualifying_position,q1,q2,q3


Missing Q2 / Q3 per race (expected about 5 / 10 in the knock-out format):


round,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
q1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
q2,5,5,4,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
q3,10,10,9,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10


## 8. Save pilot outputs

Tables go to `data/interim/` (**not** `data/processed/`), named `{season}_{table}.csv`. Missing API fields are written as empty cells. The race summary and status combinations go to `outputs/tables/`.

In [19]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

saved_files = {}
for name, df in tables.items():
    path = INTERIM_DIR / f"{SEASON}_{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    saved_files[path] = df
for name, df in [("race_summary", summary), ("status_combinations", status_combinations)]:
    path = TABLES_DIR / f"pilot_{SEASON}_{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    saved_files[path] = df

pd.DataFrame({"file": [p.relative_to(PROJECT_ROOT).as_posix() for p in saved_files],
              "rows": [len(df) for df in saved_files.values()],
              "size_kb": [round(p.stat().st_size / 1024, 1) for p in saved_files]})

,file,rows,size_kb
0,data/interim/2024_races.csv,24,3.2
1,data/interim/2024_results.csv,479,115.7
2,data/interim/2024_qualifying.csv,479,97.8
3,data/interim/2024_sprints.csv,120,28.3
4,data/interim/2024_drivers.csv,25,1.5
5,data/interim/2024_constructors.csv,10,0.4
6,outputs/tables/pilot_2024_race_summary.csv,24,1.4
7,outputs/tables/pilot_2024_status_combinations.csv,10,0.3


### 8.1 Re-run check: are cached responses reused, and are outputs identical?

We create a **new** client and collect the season again. It must make **0** network requests, and every table must be identical to the first run, down to the CSV bytes.

In [20]:
rerun_client = JolpicaClient()
rerun_tables = collect_season(rerun_client, SEASON)

print("Re-run client activity:", rerun_client.stats)
for name, df in tables.items():
    saved_bytes = (INTERIM_DIR / f"{SEASON}_{name}.csv").read_bytes()
    buffer = io.StringIO()
    rerun_tables[name].to_csv(buffer, index=False)
    same_csv = buffer.getvalue().encode("utf-8").replace(b"\r\n", b"\n") == saved_bytes.replace(b"\r\n", b"\n")
    print(f"  {name:<13} identical DataFrame={rerun_tables[name].equals(df)}  identical CSV={same_csv}")

Re-run client activity: {'cache_hits': 15, 'network_requests': 0, 'retries': 0}
  races         identical DataFrame=True  identical CSV=True
  results       identical DataFrame=True  identical CSV=True
  qualifying    identical DataFrame=True  identical CSV=True
  sprints       identical DataFrame=True  identical CSV=True
  drivers       identical DataFrame=True  identical CSV=True
  constructors  identical DataFrame=True  identical CSV=True


### 8.2 Raw data preservation

In [21]:
raw_after = snapshot_raw()
changed = [f for f in raw_before if f in raw_after and raw_before[f] != raw_after[f]]
removed = [f for f in raw_before if f not in raw_after]
added = sorted(set(raw_after) - set(raw_before))
manifest = pd.read_csv(MANIFEST)

print("Raw files before:", len(raw_before), "| after:", len(raw_after), "| added this run:", len(added))
print("Existing raw files changed:", len(changed), "| removed:", len(removed))
print("Manifest rows added this run:", len(manifest) - manifest_rows_before,
      "| network requests this run:", client.stats["network_requests"] + rerun_client.stats["network_requests"])
print("\nManifest entries for the 2024 season-level downloads (first download time is kept):")
manifest[manifest["path"].str.match(rf"^/{SEASON}/[a-z]+\.json$")][["fetched_at_utc", "path", "params", "status", "mrdata_total"]]

Raw files before: 18 | after: 18 | added this run: 0
Existing raw files changed: 0 | removed: 0
Manifest rows added this run: 0 | network requests this run: 0

Manifest entries for the 2024 season-level downloads (first download time is kept):


,fetched_at_utc,path,params,status,mrdata_total
3,2026-09-25T14:13:48Z,/2024/races.json,"{""limit"": 100, ""offset"": 0}",200,24
4,2026-09-25T14:13:49Z,/2024/results.json,"{""limit"": 100, ""offset"": 0}",200,479
5,2026-09-25T14:13:50Z,/2024/results.json,"{""limit"": 100, ""offset"": 100}",200,479
6,2026-09-25T14:13:51Z,/2024/results.json,"{""limit"": 100, ""offset"": 200}",200,479
7,2026-09-25T14:13:52Z,/2024/results.json,"{""limit"": 100, ""offset"": 300}",200,479
8,2026-09-25T14:13:53Z,/2024/results.json,"{""limit"": 100, ""offset"": 400}",200,479
9,2026-09-25T14:13:54Z,/2024/qualifying.json,"{""limit"": 100, ""offset"": 0}",200,479
10,2026-09-25T14:13:55Z,/2024/qualifying.json,"{""limit"": 100, ""offset"": 100}",200,479
11,2026-09-25T14:13:56Z,/2024/qualifying.json,"{""limit"": 100, ""offset"": 200}",200,479
12,2026-09-25T14:13:58Z,/2024/qualifying.json,"{""limit"": 100, ""offset"": 300}",200,479


## 9. Final pilot summary

In [22]:
pilot_summary = pd.Series({
    "races discovered": len(races),
    "sprint weekends": int(races["sprint_scheduled"].sum()),
    "race-result rows": len(results),
    "qualifying rows": len(qualifying),
    "sprint rows": len(sprints),
    "unique drivers (results)": results["driver_id"].nunique(),
    "drivers in season list": len(drivers),
    "unique constructors (results)": results["constructor_id"].nunique(),
    "duplicate (season, round, driver_id) keys": int(results.duplicated(KEY).sum() + qualifying.duplicated(KEY).sum() + sprints.duplicated(KEY).sum()),
    "missing IDs (all tables)": int(missing.values.sum()),
    "grid = 0 rows (race)": int((pd.to_numeric(results["grid"]) == 0).sum()),
    "network requests (this run)": client.stats["network_requests"] + rerun_client.stats["network_requests"],
    "cache hits (this run)": client.stats["cache_hits"] + rerun_client.stats["cache_hits"],
})
pilot_summary.to_frame("value")

,value
races discovered,24
sprint weekends,6
race-result rows,479
qualifying rows,479
sprint rows,120
unique drivers (results),24
drivers in season list,25
unique constructors (results),10
"duplicate (season, round, driver_id) keys",0
missing IDs (all tables),0


### 9.2 Findings (first run 2026-09-25; every statement is observed in the outputs above)

**Collection**
- The 2024 calendar from the API has **24 races** (rounds 1–24, in order, no duplicate rounds or race_ids, `race_time` present for all). Sprint weekends: rounds **5, 6, 11, 19, 21, 23**. Calendar and sprint results agree.
- First run: **15 API requests**. Calendar 1, results 5 pages, qualifying 5 pages, sprint 2 pages, drivers 1, constructors 1. Each download is logged in `data/raw/_manifest.csv`.
- Every table's row count equals `MRData.total`: results **479**, qualifying **479**, sprint **120**, drivers 25, constructors 10.
- The season-level (paginated) data is **identical** to the per-round responses cached in Step 3 (R1 results, R1 qualifying, R5 sprint).

**Keys and IDs**
- **0** duplicate `(season, round, driver_id)` keys in results, qualifying and sprint. **0** missing race, driver or constructor IDs.
- 24 drivers and 10 constructors appear in the results. Every results driver and constructor is in the season lists.
- **`antonelli` is in the 2024 season driver list but has no race, qualifying or sprint record.** The season driver list is therefore not the same as "drivers who raced". The API does not give the reason.
- In every race, the qualifying driver set is identical to the results driver set.

**Record counts**
- 23 races have 20 results. **Australian GP (R3) has 19**: `sargeant` has no record at all and Williams has one entry (`albon`). A driver who did not take part has **no row**; he is not a "W" row. The API does not state the reason.
- Qualifying Q2/Q3 are absent for 5/10 drivers per race (9 in R3 because of 19 drivers). Q1 is present for all 479 rows. Missing values stay empty; nothing is invented.

**Positions, grid and status (race results)**
- `position` is numeric for all rows, within 1..N, and never repeated within a race. `grid` is numeric and within 0..N.
- **grid = 0: 1 row**: `ocon`, Mexico City GP (R20), finished 13th, status Lapped (as in Step 2).
- Status × positionText (race): Finished/numeric 287 · Lapped/numeric 138 · Retired/R 43 · **Retired/numeric 6** · Did not start/W 3 · Disqualified/D 1 · **Disqualified/numeric 1**.
- Sprint: Finished/numeric 116 · Retired/R 3 · **Retired/numeric 1** (`alonso`, R5, 17 of 19 laps).
- Disqualified: `russell` (R14, positionText "20", 44 laps) and `hulkenberg` (R21, "D", 0 laps). Did not start: `gasly` (R12), `albon` and `stroll` (R21), all with positionText W, 0 laps and a **non-zero grid**.
- **No DNF rule was applied.** These combinations are the input for the Step 7 decision.

**Re-run and preservation**
- A second collection with a new client made **0 network requests** (15 cache hits) and produced **identical** DataFrames and CSV bytes for all 6 tables.
- Raw files before → after: 3 → 18 (15 added). **0 existing raw files changed or removed.** Manifest rows added = network requests (15).

**Still open for later steps:** the DNF/DSQ/non-starter rule (Step 7); whether absent drivers (like R3 `sargeant`) and list-only drivers (like `antonelli`) occur in other seasons (full collection); grid = 0 meaning in older seasons (Step 7).